In [2]:
pip install requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
from datetime import date
import requests
import pandas as pd

# ==== GANTI BARIS DI BAWAH INI DENGAN API KEY KAMU ====
FRED_API_KEY = "78e37a8eb3f1bd40fc6abf986766ea24"
# ========================================================

BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
START_DATE = "2015-01-01"
SERIES_IDS = ["DFEDTARU", "DFEDTARL"]
OUTPUT_FILE = "fomc_target_range_2015_present.csv"

def fetch_series(series_id, api_key):
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": START_DATE,
        "observation_end": date.today().isoformat(),
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if "observations" not in data:
        raise RuntimeError(f"Response tidak valid untuk {series_id}: {data}")
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df = df.rename(columns={"value": series_id})
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df

if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY kosong.")

print("Mengambil data dari FRED...")
merged = None
for sid in SERIES_IDS:
    df = fetch_series(sid, FRED_API_KEY)
    print(f"  {sid}: {len(df)} baris diambil")
    merged = df if merged is None else merged.merge(df, on="date", how="outer")

merged = merged.sort_values("date").reset_index(drop=True)
changed = (merged["DFEDTARU"].diff() != 0) | (merged["DFEDTARL"].diff() != 0)
merged["rate_changed"] = changed.fillna(True)
merged.to_csv(OUTPUT_FILE, index=False)

print(f"\nSelesai. {len(merged)} baris disimpan ke: {OUTPUT_FILE}")
print(merged.tail())

Mengambil data dari FRED...
  DFEDTARU: 4249 baris diambil
  DFEDTARL: 4249 baris diambil

Selesai. 4249 baris disimpan ke: fomc_target_range_2015_present.csv
            date  DFEDTARU  DFEDTARL  rate_changed
4244  2026-08-15      3.75       3.5         False
4245  2026-08-16      3.75       3.5         False
4246  2026-08-17      3.75       3.5         False
4247  2026-08-18      3.75       3.5         False
4248  2026-08-19      3.75       3.5         False
